# VoxPrep HK — Kaggle QLoRA training and export

**Unexecuted notebook: no training or human review is claimed.** Use Kaggle first; free Colab is a fallback and GPU availability is not guaranteed.

Before running: complete Kaggle **phone verification**, enable **Internet**, and select a CUDA GPU in Session options. A T4 is sufficient for the intended QLoRA setup; the script uses GPU 0 only even if two T4s appear. T4 does **not** support BF16: `ml/train.py` detects this and uses FP16. Qwen3-4B weights require roughly **8 GB** of downloads, plus CUDA packages/cache, adapters/checkpoints and conversion space. Check free disk space and the account's displayed GPU quota. No paid compute is requested.

Commit the real, merged `ml/data/reviewed_examples.jsonl` to the public fork branch selected below **after** WS1/WS2/WS3/WS5 are integrated. Only that source file is needed; this notebook makes seed-42 splits. Never fabricate reviews or turn off data gates. Do not put private applicant/employer information or tokens in Git.

References: [Kaggle notebooks](https://www.kaggle.com/docs/notebooks), [phone/GPU access](https://www.kaggle.com/product-feedback/268333), [free Colab availability](https://research.google.com/colaboratory/faq.html).

## 1. Clone a specific public fork branch

Set `BRANCH` to the published branch containing all implementation changes and the final human-reviewed corpus. The default is the fork's `main`, **not** the upstream repository. An existing checkout must match the requested URL/branch and be clean; this cell does not overwrite it. The exact source commit is recorded for reproducibility. No manual source upload or Kaggle API token is needed.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/kelvin0611/COMP4431_GroupProject_InterviewCoach.git"
BRANCH = "main"  # Change if reviewed training data is published on another branch.
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
PROJECT = WORK / "COMP4431_GroupProject_InterviewCoach"
WORK.mkdir(parents=True, exist_ok=True)
if not PROJECT.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--single-branch",
            "--branch",
            BRANCH,
            REPO,
            str(PROJECT),
        ],
        check=True,
    )
else:
    assert (
        subprocess.check_output(
            ["git", "-C", str(PROJECT), "remote", "get-url", "origin"], text=True
        ).strip()
        == REPO
    )
    assert (
        subprocess.check_output(
            ["git", "-C", str(PROJECT), "branch", "--show-current"], text=True
        ).strip()
        == BRANCH
    )
    assert not subprocess.check_output(
        ["git", "-C", str(PROJECT), "status", "--porcelain"], text=True
    ).strip(), "Existing checkout has edits; inspect them before continuing."
os.chdir(PROJECT)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
SOURCE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Training source:", REPO, BRANCH, SOURCE_COMMIT)
assert (PROJECT / "ml/data/reviewed_examples.jsonl").is_file(), (
    "Publish the real merged human-reviewed corpus on BRANCH first. Do not use drafts or create approvals here."
)

## 2. Install the recorded Python/CUDA dependencies

Install the repository's software dependencies and pinned training stack. Commands use `check=True`: a failed installation or data validation stops the cell. If the notebook asks for a kernel restart, restart and rerun the clone cell before continuing. GPU training has not been executed during development of this notebook.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.lock"], check=True
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "ml/requirements-train.txt"],
    check=True,
)

## 3. Validate review evidence and create seed-42 splits

Every row requires current human approval; the assigned stratified sample requires two independent reviewers. Duplicates and cross-split source/template/role groups are rejected. The existing splitter records seed 42. Confirm the real corpus count matches `expected_examples` (planned WS2 count: **480**). For an intentionally smaller reviewed corpus, change the committed config beforehand and report the actual size; do not silently resize it here. The 2,048-token and assistant-only masking gates remain in `train.py`.

In [ ]:
import json

config = json.loads(Path("ml/config.json").read_text())
assert config["seed"] == 42, "This notebook's split protocol uses seed 42."
subprocess.run(
    [
        sys.executable,
        "-m",
        "ml.dataset",
        "validate",
        "--input",
        "ml/data/reviewed_examples.jsonl",
        "--require-review",
        "--check-duplicates",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "ml.dataset",
        "split",
        "--input",
        "ml/data/reviewed_examples.jsonl",
        "--output",
        config["data_dir"],
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "ml.train", "--config", "ml/config.json", "--validate-only"],
    check=True,
)

## 4. Confirm CUDA and train

The configuration uses NF4 double quantization, rank 32/alpha 64, batch 1 × accumulation 16, evaluation/checkpointing every 10 optimizer steps, logging every 2, and best validation checkpoint selection. T4 uses FP16 automatically; BF16-capable GPUs use BF16. There is no CPU fallback. Set `RESUME` only to a retained checkpoint from this same run/config if recovering from a disconnection. Keep all failures and actual logs; a loss curve alone does not establish model quality.

In [ ]:
import torch

assert torch.cuda.is_available(), "Select a CUDA GPU in Kaggle/Colab session options."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Precision:", "bf16" if torch.cuda.is_bf16_supported() else "fp16")
RESUME = None  # Example: ml/runs/voxprep-qlora/checkpoints/checkpoint-20
command = [sys.executable, "-m", "ml.train", "--config", "ml/config.json"]
if RESUME:
    command += ["--resume-from-checkpoint", RESUME]
subprocess.run(command, check=True)
run = Path(config["output_dir"])
(run / "source_revision.json").write_text(
    json.dumps({"repo": REPO, "branch": BRANCH, "commit": SOURCE_COMMIT}, indent=2)
)

## 5. Inspect genuine evidence

The completed manifest, adapter hashes, loss plot and log must actually exist. Inspect the curves rather than inserting example results. The manifest is the source of training status; without a completed manifest the app stays `not_trained`.

In [ ]:
import hashlib

from IPython.display import Image, display

manifest = json.loads((run / "model_manifest.json").read_text())
assert manifest["status"] == "training_completed"
for name, digest in manifest["adapter_sha256"].items():
    assert hashlib.sha256((run / "adapter" / name).read_bytes()).hexdigest() == digest
assert (run / "training_log.jsonl").stat().st_size > 0
display(Image(filename=str(run / "loss.png")))
print("Best validation loss:", manifest["best_validation_loss"])

## 6. Export for the app — default route B

**Diagnostic route-A preflight:** official llama.cpp supports `convert_lora_to_gguf.py … --base … --outfile … --outtype f16` and `llama-server -m base.gguf --lora adapter.gguf`. Converter `--base` needs the matching pinned HF configuration, not base weights. Hot LoRA and quantized inference are supported upstream, but this project's actual Qwen adapter + Q4_K_M parity is **unverified**: development has no adapter, llama-server or converter dependencies. We could not start an end-to-end test within the plan's 30-minute cap; route B is selected. Do not label A verified merely because the flags exist.

`EXPORT_MODE = "B"` reloads the pinned base in FP16, merges the verified adapter, converts to F16 GGUF, then quantizes to Q4_K_M **on Kaggle**. Only download the final GGUF (roughly 2.5 GB; inspect actual size) plus the run archive. The disposable intermediate merged/F16 files are removed only after successful conversion/checksums to fit notebook output limits. For A after a real smoke test, choose `"A"`; it saves only the small GGUF adapter. Use `"skip"` to package evidence without conversion.

The official llama.cpp checkout revision is recorded. To repeat an export, set `LLAMA_CPP_REVISION` to that recorded commit. Build CPU-only conversion tools so it also works on T4 without CUDA compiler setup. Export is unexecuted here and can take substantial time/disk space.

References: [converter](https://github.com/ggml-org/llama.cpp/blob/master/convert_lora_to_gguf.py), [server flags](https://github.com/ggml-org/llama.cpp/blob/master/tools/server/README.md), [LoRA inference test](https://github.com/ggml-org/llama.cpp/blob/master/tests/test-lora-conversion-inference.sh).

In [ ]:
EXPORT_MODE = (
    "B"  # "A" only after real parity testing; "skip" keeps training evidence only.
)
LLAMA_CPP_REVISION = "master"  # Replace with recorded commit for repeat exports.
assert EXPORT_MODE in {"A", "B", "skip"}
if EXPORT_MODE != "skip":
    llama = WORK / "llama.cpp"
    if not llama.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/ggml-org/llama.cpp.git",
                str(llama),
            ],
            check=True,
        )
    subprocess.run(
        [
            "git",
            "-C",
            str(llama),
            "fetch",
            "--depth",
            "1",
            "origin",
            LLAMA_CPP_REVISION,
        ],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(llama), "checkout", "--detach", "FETCH_HEAD"], check=True
    )
    llama_revision = subprocess.check_output(
        ["git", "-C", str(llama), "rev-parse", "HEAD"], text=True
    ).strip()
    (run / "llama_cpp_revision.txt").write_text(llama_revision + "\n")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-r",
            str(llama / "requirements/requirements-convert_hf_to_gguf.txt"),
        ],
        check=True,
    )
    if EXPORT_MODE == "A":
        from huggingface_hub import snapshot_download

        pinned_config = snapshot_download(
            manifest["base_model"],
            revision=manifest["base_revision"],
            allow_patterns=["config.json", "tokenizer*", "special_tokens_map.json"],
        )
        adapter_gguf = run / "adapter.gguf"
        subprocess.run(
            [
                sys.executable,
                str(llama / "convert_lora_to_gguf.py"),
                str(run / "adapter"),
                "--base",
                pinned_config,
                "--outfile",
                str(adapter_gguf),
                "--outtype",
                "f16",
            ],
            check=True,
        )
        (run / "adapter_gguf_sha256.txt").write_text(
            hashlib.sha256(adapter_gguf.read_bytes()).hexdigest() + "\n"
        )
        print(
            "Route A conversion complete; real base+adapter inference validation still required."
        )
    else:
        subprocess.run(
            [
                "cmake",
                "-S",
                str(llama),
                "-B",
                str(llama / "build"),
                "-DGGML_CUDA=OFF",
                "-DLLAMA_CURL=OFF",
            ],
            check=True,
        )
        subprocess.run(
            [
                "cmake",
                "--build",
                str(llama / "build"),
                "--target",
                "llama-quantize",
                "-j",
                "2",
            ],
            check=True,
        )
        export_dir = WORK / "voxprep-export"
        subprocess.run(
            [
                sys.executable,
                "-m",
                "ml.export",
                "--run",
                str(run),
                "--output",
                str(export_dir),
                "--llama-cpp",
                str(llama),
                "--quantize-bin",
                str(llama / "build/bin/llama-quantize"),
            ],
            check=True,
        )
        exported = json.loads((export_dir / "export_manifest.json").read_text())
        final_gguf = export_dir / "voxprep-Q4_K_M.gguf"
        digest = hashlib.sha256()
        with final_gguf.open("rb") as file:
            for block in iter(lambda: file.read(1024 * 1024), b""):
                digest.update(block)
        assert digest.hexdigest() == exported["files"][final_gguf.name]
        import shutil

        shutil.copy2(export_dir / "export_manifest.json", run / "export_manifest.json")
        # Only disposable artifacts created by this export, after final checksum validation.
        shutil.rmtree(export_dir / "merged")
        (export_dir / "voxprep-f16.gguf").unlink()
        print(
            "Download final GGUF separately:",
            final_gguf,
            "GiB:",
            round(final_gguf.stat().st_size / 2**30, 2),
        )

## 7. Download evidence and connect the app

The last code cell archives the entire run: `adapter/`, `loss.png`, `training_log.jsonl`, `model_manifest.json`, split/config/environment records, checkpoints and export/source revision evidence. Kaggle exposes the ZIP and (route B) GGUF under **Output**; save a notebook version with outputs and download both. Colab triggers a ZIP download and provides a GGUF link.

Extract the archive **into** `ml/runs/voxprep-qlora/` locally. Keep the base model separately for evaluation.

- **B:** set `MODEL_PROVIDER=llama` and `MODEL_PATH=models/voxprep-Q4_K_M.gguf`; leave `LORA_PATH` unset. Run `make dev` or `make demo` after stopping your previous model service explicitly.
- **A (after smoke validation):** set `MODEL_PATH` to the matching base Q4_K_M and `LORA_PATH=ml/runs/voxprep-qlora/adapter.gguf`. The launcher checks the adapter exists and refuses to reuse a running service without it active.
- Run `python -m evaluation.metadata` to refresh the saved training fields. The Methodology endpoint also reads the actual local manifest; missing/incomplete manifests keep `not_trained`. `adapter_available` separately indicates local artifact presence. Training completion is not an evaluation pass.
- Follow `docs/EVALUATION.md` / WS4 for the **entire real held-out test split**, with actual sample counts. Preserve base vs fine-tuned failures and human ratings; do not pad to 120 prompts or use training examples.

In [ ]:
import shutil

from IPython.display import FileLink, display

required = [
    run / "adapter/adapter_model.safetensors",
    run / "loss.png",
    run / "training_log.jsonl",
    run / "model_manifest.json",
]
assert all(path.is_file() for path in required), (
    "Incomplete run: do not package as completed."
)
archive = shutil.make_archive(
    str(WORK / "voxprep-training-evidence"), "zip", root_dir=run
)
print("Download and retain:", archive)
display(FileLink(os.path.relpath(archive, WORK)))
if EXPORT_MODE == "B":
    display(FileLink(os.path.relpath(final_gguf, WORK)))
try:
    from google.colab import files

    files.download(archive)
except ImportError:
    print("Kaggle: save the notebook version and download ZIP/GGUF from Output.")